# Pengemasan encoder: berbagi bobot antar-function

Kandidat saat ini memakai **frontend CPU + encoder window penuh CPU+GPU + partial CPU**. Alleluia selesai 33,44s dan Raise a Hallelujah 92,69s, seluruh label sesuai Mac dan selisih probabilitas akhir di bawah `1e-4`. Ini hasil dua run CPU-control; cache/load memengaruhi angka waktu. GPU frontend ditahan setelah selisih probabilitas pada uji sebelumnya melewati batas.

Sekarang ukur pengurangan ukuran dengan Core ML multi-function. Untuk setiap encoder, gabungkan function `short_30`, `long_120`, dan `partial` dalam satu `.mlpackage`, berbagi konstanta yang identik. Tidak ada training, perubahan bobot, kuantisasi ulang, perubahan durasi konteks atau boundary. Core ML menghapus duplikasi konstanta; tidak semua bobot antar-export dijamin identik, sehingga penghematan harus diukur.

Script memeriksa hash graph komputasi yang mengabaikan nama variabel tetapi mempertahankan aliran data, nilai konstanta, dtype dan shape; payload terkompresi serta kontrak input/output juga harus sama. Di Mac, kedua profil penuh dan empat panjang partial diuji terhadap sumber pada CPU, lalu paket dikompilasi. Input partial berasal dari prefix fixture feature nyata; pemeriksaan ini belum full validation sesudah packaging atau pengujian GPU iPhone.

Output hanya di `outputs.local`. Model aplikasi dan sumber tidak ditimpa. MusicFM dan MuQ dijalankan dalam proses terpisah agar memori proses dilepas antar-encoder. Paket hasil belum dipasang ke SongMap.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib,json,subprocess
ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/package_encoder_weights.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python venv tidak ditemukan: {RUN_PYTHON}'  # Jangan resolve symlink executable.
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED_SELECTION='f65edb95bae3fb2a948d742c647168763217180fb67b68ab5d82ad9ca5fc491d'
EXPECTED_REPAIR='3e84fe326e03a8f0e0eed1bff07c1fe00ea15cbf7ed3175f70c8fa9e14e408c0'
quality=[]
for path in (ROOT/'outputs.local').glob('encoder-glu-quality-*/validation-all.json'):
    r=json.loads(path.read_text())
    if (r.get('selection_identity')==EXPECTED_SELECTION and r.get('graph_repair_sha256')==EXPECTED_REPAIR
            and r.get('full_validation') and r.get('section_agreement')==1):quality.append(path)
assert quality, 'Full validation GLU repair yang sesuai belum ditemukan'
QUALITY=max(quality,key=lambda p:p.stat().st_mtime).resolve()
REPAIR=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-glu-repair-*/repair-report.json') if sha(p)==EXPECTED_REPAIR),None)
assert REPAIR, 'Repair yang sesuai belum ditemukan'
RESUME_OUTPUT=None  # Isi folder encoder-weight-packaging-* sebelumnya setelah restart kernel.
OUTPUT=(Path(RESUME_OUTPUT).expanduser().resolve() if RESUME_OUTPUT else
        ROOT/'outputs.local'/('encoder-weight-packaging-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f')))
REPORTS={}
print('Quality:',QUALITY)
print('Repair:',REPAIR)
print('Output:',OUTPUT)

def package(role):
    # Hasil lengkap dapat dipakai ulang; pemasangan di device nantinya tetap memeriksa hash paket.
    done=[]
    for path in OUTPUT.glob(role+'-*/packaging-report.json'):
        r=json.loads(path.read_text())
        if (r.get('protocol')=='encoder-shared-weights-v1' and r.get('encoder')==role and r.get('native_cpu_passed')
                and r.get('compiled_digest') and not r.get('error')
                and r.get('script_sha256')==sha(ROOT/'scripts/package_encoder_weights.py')
                and r['evidence']['quality_sha256']==sha(QUALITY) and r['evidence']['repair_sha256']==sha(REPAIR)):
            done.append(path)
    if done:
        REPORTS[role]=max(done,key=lambda p:p.stat().st_mtime)
        print('Report selesai digunakan kembali:',REPORTS[role]);return
    attempt=OUTPUT/(role+'-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
    log=Path(str(attempt)+'.log');log.parent.mkdir(parents=True,exist_ok=True)
    args=[RUN_PYTHON,'-u',ROOT/'scripts/package_encoder_weights.py',
          '--songstruct-root',SONGSTRUCT_ROOT,'--songmap-root',ROOT,'--quality-report',QUALITY,
          '--graph-repair',REPAIR,'--encoder',role,'--output-dir',attempt]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen([str(x) for x in args],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Packaging {role} gagal ({status}); lihat {log}. Source dan hasil encoder lain tetap tersimpan. Ulang cell ini memakai folder attempt baru.')
    REPORTS[role]=attempt/'packaging-report.json'


In [ ]:
package('musicfm')


In [ ]:
package('muq')


In [ ]:
from IPython.display import display,Markdown
rows=[]
for role in ('musicfm','muq'):
    path=REPORTS[role];r=json.loads(path.read_text())
    assert r['native_cpu_passed'] and all(c['passed'] for c in r['native_checks'])
    assert all(all(c.values()) for c in r['checks'].values())
    rows.append(r)
    print(role,'candidate for device test:',r['candidate_for_device_test'],'| report:',path)
lines=['| Encoder | 3 paket sumber, raw MiB | Paket berbagi, raw MiB | Pengurangan raw | Compiled MiB |',
       '|---|---:|---:|---:|---:|']
for r in rows:
    lines.append(f"| {r['encoder']} | {r['source_bytes']/2**20:.1f} | {r['packed_bytes']/2**20:.1f} | {r['size_reduction_fraction']:.1%} | {r['compiled_bytes']/2**20:.1f} |")
display(Markdown('\n'.join(lines)))
summary=dict(protocol='encoder-shared-weights-summary-v1',reports={role:dict(path=str(path),sha256=sha(path)) for role,path in REPORTS.items()},
             source_encoder_raw_bytes=sum(r['source_bytes'] for r in rows),packed_encoder_raw_bytes=sum(r['packed_bytes'] for r in rows),
             packed_encoder_compiled_bytes=sum(r['compiled_bytes'] for r in rows),ready_for_app_integration=False)
(OUTPUT/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print('Kirim kedua packaging-report.json atau ringkasannya:',OUTPUT/'summary.json')
print('Ukuran ini encoder saja. Frontend/head/context tidak dihitung; ukuran aplikasi terpasang belum diukur.')


Jika semua pemeriksaan lulus dan ukuran mengecil, tahap berikutnya menghubungkan function yang benar ke route yang telah dipilih dan menguji pipeline di iPhone. Penggabungan fungsi dapat mengubah cara compiler native memuat model, sehingga stabilitas/latensi/memori harus diperiksa lagi sebelum mengganti aset produksi.

Tidak ada janji angka penghematan atau percepatan sebelum hasil model asli tersedia. Ukuran raw dibandingkan dengan raw; compiled dilaporkan terpisah. Pengemasan ini mengurangi duplikasi penyimpanan, bukan otomatis mengurangi jumlah operasi inferensi.

Hasil pengemasan yang sudah ditinjau hanya menghemat 4,10% total raw encoder. Lanjutkan [encoder_sharing_audit.ipynb](encoder_sharing_audit.ipynb) untuk audit read-only penyebab sisa ukuran sebelum pemasangan atau perubahan bobot.
